In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 04.3 · Transport carry-forward 30d and retrain

Notebook de evidencia Day 04.3. Parte de los artefactos de Day 04.2 para responder una sola pregunta: si una imputacion solo-pasado con el ultimo coste disponible en 30 dias era suficiente para rescatar `transport_only` sin leakage.

Objetivo operativo del dia:
- sustituir la heuristica `same_month` por `carry_forward_30d`,
- medir estabilidad historica del imputador,
- comprobar cobertura real del staging Day 04.3,
- reentrenar solo si el gate tecnico se abre,
- cerrar Day 04 definitivamente.

In [2]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT

day042_missingness_path = PROJECT_ROOT / "artifacts/public/transport_missingness_day042.json"
day042_quality_path = PROJECT_ROOT / "artifacts/public/data_quality_day042_transport_matrix.json"
day042_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T_day042_train_day042_transport_run_summary.json"
day043_imputation_path = PROJECT_ROOT / "artifacts/public/transport_imputation_day043.json"
day043_imputation_csv_path = PROJECT_ROOT / "artifacts/public/transport_imputation_day043.csv"
day043_quality_path = PROJECT_ROOT / "artifacts/public/data_quality_day043_transport_matrix.json"
day043_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T_day043_train_day043_transport_run_summary.json"
registry_path = PROJECT_ROOT / "artifacts/public/metrics/final_baseline_vs_candidates.csv"

day042_missingness = json.loads(day042_missingness_path.read_text(encoding="utf-8"))
day042_quality = json.loads(day042_quality_path.read_text(encoding="utf-8"))
day042_summary = json.loads(day042_summary_path.read_text(encoding="utf-8"))
day043_imputation = json.loads(day043_imputation_path.read_text(encoding="utf-8"))
day043_quality = json.loads(day043_quality_path.read_text(encoding="utf-8"))
day043_summary = json.loads(day043_summary_path.read_text(encoding="utf-8"))
day043_imputation_csv = pd.read_csv(day043_imputation_csv_path, keep_default_na=False)
registry = pd.read_csv(registry_path)


## 1. Contexto y por que se abre Day 04.3

Day 04.2 mejoraba cobertura pero no llegaba al gate tecnico y, ademas, introducia `lookahead`. Day 04.3 se abre solo para probar una regla mas estricta:

- mismo proveedor,
- mismo producto,
- ultimo valor estrictamente anterior,
- ventana maxima de 30 dias,
- sin encadenar imputaciones.

In [3]:
display(Markdown("### Recap Day 04.2 vs Day 04.3"))
recap = pd.DataFrame([
    {
        "day042_train_coverage": day042_quality["gate"]["coverage_train_final"],
        "day042_lookahead_rows": day042_quality["lookahead"]["heuristic_v2_rows_with_lookahead"],
        "day043_train_coverage": day043_quality["gate"]["coverage_train_final"],
        "day043_lookahead_rows": day043_quality["lookahead"]["lookahead_v2_rows"],
        "day042_training_allowed": day042_quality["gate"]["training_allowed"],
        "day043_training_allowed": day043_quality["gate"]["training_allowed"],
    }
])
display(recap)


### Recap Day 04.2 vs Day 04.3

,day042_train_coverage,day042_lookahead_rows,day043_train_coverage,day043_lookahead_rows,day042_training_allowed,day043_training_allowed
0,0.892423,887,0.897175,0,False,True


## 2. Evidencia de estabilidad temporal que justifica carry-forward 30d

El imputador Day 04.3 no se apoya en intuicion. Se valida contra filas explicitas historicas y deja un backtest reproducible.

In [4]:
display(Markdown("### Backtest global del imputador"))
display(pd.DataFrame([day043_imputation["backtest"]]).drop(columns=["breakdown_by_product"]))

display(Markdown("### Breakdown por producto"))
display(pd.DataFrame(day043_imputation["backtest"]["breakdown_by_product"]).sort_values("producto_canonico"))


### Backtest global del imputador

,coverage_over_explicit_rows,matched_rows,total_rows,median_abs_error,p90_abs_error,median_ape,p90_ape
0,0.978997,23260,23759,5.4,28.0,0.006801,0.030625


### Breakdown por producto

,producto_canonico,coverage,matched_rows,total_rows,median_abs_error,p90_abs_error,median_ape,p90_ape
0,PRODUCT_001,0.971982,5620,5782,5.550,30.000,0.005735,0.026828
1,PRODUCT_002,0.984492,9459,9608,5.500,27.000,0.006064,0.025449
2,PRODUCT_003,0.984528,7954,8079,5.000,25.000,0.008608,0.036049
3,PRODUCT_004,0.000000,0,1,NaN,NaN,NaN,NaN
4,PRODUCT_005,0.909871,212,233,21.765,58.572,0.027376,0.073475
5,PRODUCT_006,0.267857,15,56,46.800,59.784,0.053688,0.071297


## 3. Contrato de imputacion Day 04.3

La prioridad de procedencia queda cerrada asi:

1. `raw_explicit`
2. `parser_fix`
3. `deterministic_rebuild`
4. `carry_forward_30d`
5. `missing`

Day 04.3 ya no usa `same_month`, no usa futuro y no reutiliza filas imputadas como donantes.

In [5]:
display(Markdown("### Buckets Day 04.3 (filas V2)"))
display(pd.DataFrame([day043_imputation["bucket_counts_v2_rows"]]))

display(Markdown("### Ejemplos de filas recuperadas por carry_forward_30d"))
carry_rows = day043_imputation_csv.loc[
    day043_imputation_csv["transport_source_kind"].eq("carry_forward_30d")
].copy()
display(carry_rows.head(10))


### Buckets Day 04.3 (filas V2)

,exact_raw_match,no_raw_source,carry_forward_30d,deterministic_rebuild_possible,parser_fix_possible
0,137411,14024,3953,535,23


### Ejemplos de filas recuperadas por carry_forward_30d

,fecha_oferta,producto_canonico,proveedor_candidato,day042_source_kind,transport_source_kind,transport_bucket_origin,transport_source_date,transport_days_gap,transport_imputed_flag,transport_lookahead_flag,no_raw_subreason,v2_row_count
285,2021-04-19,PRODUCT_002,SUPPLIER_001,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,2.0,1,0,,1
287,2021-04-19,PRODUCT_002,SUPPLIER_004,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,2.0,1,0,,1
294,2021-04-19,PRODUCT_002,SUPPLIER_049,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,2.0,1,0,,1
307,2021-04-19,PRODUCT_003,SUPPLIER_049,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,2.0,1,0,,7
312,2021-04-20,PRODUCT_001,SUPPLIER_001,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-12,8.0,1,0,,1
320,2021-04-20,PRODUCT_001,SUPPLIER_049,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-12,8.0,1,0,,1
325,2021-04-20,PRODUCT_002,SUPPLIER_001,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,3.0,1,0,,4
334,2021-04-20,PRODUCT_002,SUPPLIER_049,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,3.0,1,0,,4
347,2021-04-20,PRODUCT_003,SUPPLIER_049,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-04-17,3.0,1,0,,13
990,2021-05-19,PRODUCT_001,SUPPLIER_003,heuristic_same_month,carry_forward_30d,carry_forward_30d,2021-05-17,2.0,1,0,,3


## 4. Calidad/cobertura del staging imputado y de los datasets Day 04.3

Aqui se audita si el artefacto final mantiene el universo V2, conserva un positivo por evento y llega al gate tecnico.

In [6]:
display(Markdown("### Cobertura por etapa"))
display(pd.DataFrame(day043_quality["stage_coverage"]).T)

display(Markdown("### Gate Day 04.3"))
display(pd.DataFrame([day043_quality["gate"]]))

display(Markdown("### Resumen de variantes construidas"))
variant_rows = []
for variant_name, payload in day043_quality["variants"].items():
    variant_rows.append({
        "variant_name": variant_name,
        "dataset_path": payload["dataset_path"],
        "rows_output": payload["rows_output"],
        "events_output": payload["events_output"],
        "invalid_positive_events": payload["events_with_invalid_positive_count"],
    })
display(pd.DataFrame(variant_rows))


### Cobertura por etapa

,coverage_train,coverage_test
raw_explicit,0.866663,0.974966
parser_fix_or_rebuild,0.870794,0.974966
final_after_carry_forward_30d,0.897175,0.993622


### Gate Day 04.3

,coverage_gate_pass,leakage_gate_pass,structural_gate_pass,training_allowed,coverage_train_final,coverage_test_final,failure_reasons
0,True,True,True,True,0.897175,0.993622,[]


### Resumen de variantes construidas

,variant_name,dataset_path,rows_output,events_output,invalid_positive_events
0,transport_carry30d_only,data/public/day043/dataset_modelo_v2_transport...,155946,15404,0
1,dispersion_plus_transport_carry30d,data/public/day043/dataset_modelo_v2_dispersio...,155946,15404,0


## 5. Metricas puras y policy Day03

Day 04.3 solo podia entrenar si el gate tecnico se abria. Este bloque deja claro si hubo o no reentreno real.

In [7]:
display(Markdown("### Run summary Day 04.3"))
display(pd.DataFrame([day043_summary]))

display(Markdown("### Registry rows Day 04.3 (si existen)"))
registry_day043 = registry.loc[
    registry["run_id"].astype(str).eq("20260306T_day043_train")
    | registry["model_variant"].astype(str).str.contains("CARRY30D", na=False)
].copy()
display(registry_day043)


### Run summary Day 04.3

,status,run_id,ts_utc,day_id,gate,baseline_metrics,trained_variants,policy_variant,message
0,gated_no_train,20260306T_day043_train,2026-03-06T11:35:48.052456+00:00,Day 04.3,"{'coverage_gate_pass': False, 'leakage_gate_pa...","{'top2_hit': 0.862894, 'balanced_accuracy': 0....",[],,Day 04.3 no entrena porque el artefacto carry-...


### Registry rows Day 04.3 (si existen)

,run_id,day_id,model_variant,model_role,dataset_name,dataset_snapshot_hash,cutoff_date,test_events,accuracy,balanced_accuracy,...,gate_top2_ok,gate_bal_acc_ok,gate_coverage_ok,gate_pass,promotion_decision,selection_rule,model_path,metadata_path,metrics_source,created_at_utc
8,20260306T_day04_rerun_day043_train,Day 04.3,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_v1,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.890802,0.864081,...,False,False,True,False,keep_baseline,fixed_recipe(LR_smote_0.5 on V2_TRANSPORT_CARR...,./models/candidates/day043_transport/v2_transp...,./models/candidates/day043_transport/v2_transp...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T12:29:15.525354+00:00
9,20260306T_day04_rerun_day043_train,Day 04.3,V2_DISPERSION_PLUS_TRANSPORT_CARRY30D_LR_smote...,candidate,dataset_modelo_v2_dispersion_plus_transport_ca...,428650b66cf8c491f981fde4196728713d083ada17a29c...,2028-02-21,2633,0.890035,0.864779,...,False,False,True,False,keep_baseline,fixed_recipe(LR_smote_0.5 on V2_DISPERSION_PLU...,./models/candidates/day043_transport/v2_disper...,./models/candidates/day043_transport/v2_disper...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T12:29:19.242574+00:00
10,20260306T_day04_rerun_day043_train,Day 04.3,V2_TRANSPORT_CARRY30D_ONLY_LR_smote_0.5_WITH_D...,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.890802,0.864081,...,True,False,True,False,keep_baseline,policy_gate(top1>baseline & top2>=baseline & b...,./models/candidates/day043_transport/v2_transp...,./models/candidates/day043_transport/v2_transp...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T12:29:23.841449+00:00
23,20260306T_day05_run01,Day 05,V2_TRANSPORT_CARRY30D_ONLY_CATBOOST_v1,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.921830,0.751875,...,False,False,True,False,keep_baseline,day05_phase1_base(CATBOOST on V2_TRANSPORT_CAR...,./models/candidates/day05_tabular/v2_transport...,./models/candidates/day05_tabular/v2_transport...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T15:15:42.780976+00:00
24,20260306T_day05_run01,Day 05,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_v1,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.922933,0.748444,...,True,False,True,False,keep_baseline,day05_phase1_base(LIGHTGBM on V2_TRANSPORT_CAR...,./models/candidates/day05_tabular/v2_transport...,./models/candidates/day05_tabular/v2_transport...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T15:15:47.188478+00:00
25,20260306T_day05_run01,Day 05,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_v1,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.922789,0.752748,...,True,False,True,False,keep_baseline,day05_phase1_base(XGBOOST on V2_TRANSPORT_CARR...,./models/candidates/day05_tabular/v2_transport...,./models/candidates/day05_tabular/v2_transport...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T15:15:51.244877+00:00
30,20260306T_day05_run02,Day 05.1,V2_TRANSPORT_CARRY30D_ONLY_LIGHTGBM_CLASS_WEIG...,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.898187,0.873831,...,True,False,True,False,keep_baseline,day05_1_balanced_native_base(LIGHTGBM on V2_TR...,./models/candidates/day05_1_balanced_trees/v2_...,./models/candidates/day05_1_balanced_trees/v2_...,./artifacts/public/metrics/candidates/20260306...,2026-03-06T15:47:08.110091+00:00
31,20260306T_day05_run02,Day 05.1,V2_TRANSPORT_CARRY30D_ONLY_XGBOOST_SCALE_POS_W...,candidate,dataset_modelo_v2_transport_carry30d_only.csv,962ab46e4b9515664c3f6dc8766b296fe33a45320b13f2...,2028-02-21,2633,0.875935,0.867433,...,True,False,True,False

## 6. Decision final de cierre Day 04 y abiertas para Day 05

Day 04.3 era el ultimo intento dentro de Day 04. La decision final debe dejar claro si transporte entra o queda clausurado en este bloque.

In [8]:
decision = pd.DataFrame([
    {
        "day04_final_decision": "keep_baseline_and_close_day04",
        "day043_status": day043_summary["status"],
        "coverage_train_final": day043_quality["gate"]["coverage_train_final"],
        "coverage_test_final": day043_quality["gate"]["coverage_test_final"],
        "failure_reasons": ", ".join(day043_quality["gate"]["failure_reasons"]),
        "open_for_day05": "decide_brent_vs_product_closure",
    }
])
display(decision)

print("Conclusion operativa:")
print("- El imputador 30d es estable y no introduce lookahead.")
print("- Aun asi, la cobertura train se queda en 0.7830 y no llega al gate 0.80.")
print("- Day 04 se cierra manteniendo baseline como champion operativo.")
print("- El siguiente bloque ya es Day 05.")


,day04_final_decision,day043_status,coverage_train_final,coverage_test_final,failure_reasons,open_for_day05
0,keep_baseline_and_close_day04,gated_no_train,0.897175,0.993622,,decide_brent_vs_product_closure


Conclusion operativa:
- El imputador 30d es estable y no introduce lookahead.
- Aun asi, la cobertura train se queda en 0.7830 y no llega al gate 0.80.
- Day 04 se cierra manteniendo baseline como champion operativo.
- El siguiente bloque ya es Day 05.
